# Iterators, Generators, and DataLoaders

The mechanics behind how PyTorch (and Python in general) streams data in batches: the iterator protocol, generators as a lazy shortcut for writing iterators, and building a `DataLoader` that wraps a `Dataset` to yield shuffled batches.

## Iterables: `__iter__` vs `__getitem__`

- Sequences (`list`, `tuple`, `str`, `range`) are iterables, but you don't manually track a position index yourself — `iter(seq)` returns an **iterator** object that tracks position internally, and each `next()` call advances it.
- An object is iterable via one of two protocols:
  - **`__iter__`** (modern, preferred) — returns an iterator object with `__next__`. This is what `for`/`iter()` use if present.
  - **`__getitem__`** only, no `__iter__` (old-style fallback) — Python simulates iteration by calling `obj[0]`, `obj[1]`, `obj[2]`, ... until it hits `IndexError`.
- If a class defines `__iter__`, that's used; `__getitem__` is only consulted as a fallback when `__iter__` is missing.
- An iterator is **exhausted** once it raises `StopIteration` — it can't be rewound or reused. Calling `next()` again just raises `StopIteration` again; to iterate the same data again, you need a fresh iterator (e.g. call `iter()` on the original iterable again).
- **`StopIteration`** is a built-in exception class — a signal, not an error. When `__next__` raises it, that's the standardized way an iterator says "no more values." `for` loops (and other iteration constructs) catch it automatically and just end the loop — you never see a traceback from it in normal use.
- **Common iterables used daily:** `list`, `tuple`, `str`, `dict` (iterates over keys), `set`, `range`, file objects (iterating a file yields it line by line), and `enumerate`/`zip`/`map`/`filter` (which are themselves iterators wrapping other iterables).

In [ ]:
# TODO: Implement a simple iterator class `MyRange` that mimics range(start, end):
# - takes start and end in its constructor
# - implements __iter__ so it can be used in a `for` loop
# - implements __next__ that returns the next value and raises StopIteration when done
# - once exhausted, a given instance should not restart -- iterating it again yields nothing

class MyRange:
    def __init__(self, start, end):
        self._start = start
        self._end = end
        self._index = 0
    
    def __iter__(self):
        return self
    
    def __len__(self):
        return self._end - self._start
    
    def __next__(self):
        a = self._start + self._index
        if a >= self._end:
            raise StopIteration
        self._index += 1
        return a


a = MyRange(4, 6)
while True:
    try:
        print(next(a))
    except StopIteration:
        break

print(len(a))

for i in a: # only possible bc of the itr dunder 
    print(i)

print("========")

for i in a: # only possible bc of the itr dunder 
    print(i)

4
5
2


## Iterables vs Iterators — the protocol

A **protocol** in Python is duck-typing via dunder methods: no inheritance
required. Implement the right dunders and Python's built-in functions/syntax
(`for`, `iter()`, `next()`) treat your object accordingly.

**Iterable** = implements `__iter__` (or falls back to `__getitem__`).
- `__iter__` returns a NEW iterator object (something with `__next__`).
- Its only job: hand back an iterator when `iter()` / `for` asks.
- A list is iterable but NOT its own iterator — `iter(mylist)` gives a fresh
  iterator each time, so you can loop over it repeatedly.

**Iterator** = implements `__iter__` (returning `self`) AND `__next__`.
- `__iter__` returns `self` — it IS already the iterator.
  (Needed so an iterator can be used directly in a `for` loop, since `for`
  calls `iter()` on it first.)
- `__next__` produces the next value, and raises `StopIteration` when
  exhausted — that's how `for` knows to stop.

Summary:
- Iterable's `__iter__` → returns a separate iterator.
- Iterator's `__iter__` → returns self, plus `__next__` does the work.

In [ ]:
#example of an iterable:

class MyList:
    def __init__(self, data):
        self.data = data
    def __iter__(self):
        return iter(self.data)   # returns an iterator

#example of an iterator
class MyIterator:
    def __iter__(self):
        return self          # already the iterator
    def __next__(self):
        ...                  # produce next value, or raise StopIteration

Iterable — something you can loop over. It knows how to hand you an iterator when asked. Implements __iter__, which returns a separate iterator object. A list, tuple, dict, string are all iterables.

Iterator — the thing that actually does the looping. It produces values one at a time and remembers where it is. Implements __next__ (give me the next value, or raise StopIteration when done) and __iter__ (returns self, since it's already the iterator).

A regular iterator eventually ends (raises StopIteration once exhausted). A **cyclic iterator** solves this by wrapping around and starting over once it reaches the end, looping the data indefinitely instead of stopping (e.g. `itertools.cycle`).

## Why `Dataset` and `DataLoader` are separate

- **`Dataset`** = the data collection. Implements `__len__`/`__getitem__` — random access to any single item by index. No batching or ordering logic.
- **`DataLoader`** = the iteration layer wrapped around it. Implements `__iter__`, handling batching, shuffling, and parallel loading (worker processes), yielding an iterator of batches.

The batching/shuffling/parallelism logic in `DataLoader` is generic — it only needs to call `Dataset.__getitem__(i)` for any index, it doesn't care what the items actually are. So that logic is written once and works for any dataset. If `Dataset` handled its own iteration too, every dataset class would have to reimplement shuffling, batching, and multiprocessing itself. Separating them also means the same `Dataset` can be reused with different `DataLoader` configs (batch size, shuffle on/off, train vs. eval) without touching the data-access logic.

### Summary

- PyTorch has two `Dataset` flavors: **map-style** (`__getitem__`/`__len__`) and **`IterableDataset`** (`__iter__` only). `DataLoader` adapts to whichever one it's given.
- **`__getitem__` is better suited when the data supports true random access** — you know the size up front, and any item can be fetched directly given its index, in O(1) or close to it (e.g. images on disk named/indexed by number, rows in an in-memory array, a database table with a primary key). This is what lets `DataLoader` shuffle freely: shuffling is just generating a random permutation of indices and calling `__getitem__` for each.
- **`__iter__` is better suited when the data can only be consumed sequentially** — its size may be unknown or unbounded, and there's no way to jump to item `N` without first walking through everything before it (e.g. a live stream, a huge text file with variable-length lines, a DB cursor pulling from a query in progress). Here, "shuffle" isn't a free index-permutation trick anymore — it requires buffering a window of items and sampling from that buffer, since you can't randomly seek into the underlying source.